# Module 4.3 — Recommender Systems

Netflix, Spotify, Amazon — recommendation is one of the highest-value applications of ML. The core idea connects directly to what you learned in 4.2: learn a dense embedding for every user and every item, then predict how much a user will like an item from those embeddings.

**What you'll learn:**
- The recommendation problem: explicit ratings vs implicit feedback
- Matrix factorization: the classic approach, expressed as two `nn.Embedding` layers
- Neural collaborative filtering: replace the dot product with an MLP
- Negative sampling: how to train on implicit feedback (clicks, watches)
- Making actual recommendations: top-K inference for a user
- Visualising what the learned item embeddings capture

**Dataset:** MovieLens 100K — 100,000 explicit ratings (1–5 stars) from 943 users on 1,682 movies.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
import zipfile
import pathlib

print(f'PyTorch version: {torch.__version__}')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. The Recommendation Problem

We have a matrix R where `R[u, i]` is user u's rating of item i. The matrix is **extremely sparse** — most users have only rated a tiny fraction of all movies. Our job is to fill in the missing entries.

### Explicit vs implicit feedback

| Type | Examples | What we have |
|---|---|---|
| **Explicit** | Star ratings, thumbs up/down | A real number (1–5) |
| **Implicit** | Clicks, watches, skips | Binary signal — observed or not |

Explicit feedback is cleaner to learn from (just predict the number), but rare — most users don't rate things. Implicit is abundant but noisier: a watch doesn't mean you liked it. We'll work with explicit ratings first, then discuss how implicit systems differ.

### The embedding intuition

Assign every user a vector **u** and every item a vector **v** (both length D). The predicted rating is:

```
r_hat[u, i] = u · v  +  bias_u  +  bias_i
```

If user u and item i have similar "taste vectors", their dot product is large → high predicted rating. This is called **matrix factorization** because we're approximating the full rating matrix R ≈ U × Vᵀ.

The connection to 4.2: this is exactly the same `nn.Embedding` lookup you used for words. Users and movies are tokens; we learn their representations from data.

## 3. Load MovieLens 100K

In [ ]:
data_dir = pathlib.Path('./data/ml-100k')
if not data_dir.exists():
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    print('Downloading MovieLens 100K...')
    urllib.request.urlretrieve(url, './data/ml-100k.zip')
    with zipfile.ZipFile('./data/ml-100k.zip') as zf:
        zf.extractall('./data')
    print('Done.')
else:
    print('Data already downloaded.')

ratings = pd.read_csv('./data/ml-100k/u.data', sep='\t',
                      names=['user_id', 'item_id', 'rating', 'timestamp'])
movies  = pd.read_csv('./data/ml-100k/u.item', sep='|', encoding='latin-1',
                      usecols=[0, 1], names=['item_id', 'title'])

n_users = ratings.user_id.nunique()
n_items = ratings.item_id.nunique()

print(f'Ratings: {len(ratings):,}')
print(f'Users:   {n_users:,}')
print(f'Movies:  {n_items:,}')
print(f'Rating range: {ratings.rating.min()} – {ratings.rating.max()}')
sparsity = 1 - len(ratings) / (n_users * n_items)
print(f'Matrix sparsity: {sparsity:.1%}  (most entries are missing)')
ratings.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(ratings.rating, bins=5, edgecolor='black')
axes[0].set_title('Rating distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

ratings_per_user = ratings.groupby('user_id').size()
axes[1].hist(ratings_per_user, bins=40)
axes[1].set_title('Ratings per user')
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('Number of users')

plt.tight_layout()
plt.show()
print(f'Median ratings per user: {ratings_per_user.median():.0f}')
print(f'Most active user: {ratings_per_user.max()} ratings')

## 4. Dataset and DataLoader

In [ ]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        # IDs in the file start at 1 — zero-index them for nn.Embedding
        self.users   = torch.tensor(df.user_id.values - 1, dtype=torch.long)
        self.items   = torch.tensor(df.item_id.values - 1, dtype=torch.long)
        self.ratings = torch.tensor(df.rating.values,      dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


dataset = RatingsDataset(ratings)
n_train = int(0.8 * len(dataset))
n_test  = len(dataset) - n_train
train_data, test_data = random_split(dataset, [n_train, n_test],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=512, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=512, shuffle=False)

print(f'Train: {len(train_data):,}  |  Test: {len(test_data):,}')

## 5. Matrix Factorization

Two embedding tables — one for users, one for items — plus a scalar bias for each. The prediction is their dot product plus the biases:

```
r_hat = (user_emb[u] · item_emb[i]) + user_bias[u] + item_bias[i]
```

We train with MSE loss on observed ratings. During inference we clamp predictions to [1, 5] to stay in rating range.

The bias terms matter: some users rate everything 5 stars; some movies are universally loved. The biases capture those offsets so the embeddings can focus on interaction effects.

In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, n_factors)
        self.item_emb  = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.normal_(self.user_emb.weight,  std=0.01)
        nn.init.normal_(self.item_emb.weight,  std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u    = self.user_emb(users)              # [B, n_factors]
        v    = self.item_emb(items)              # [B, n_factors]
        dot  = (u * v).sum(dim=1)               # [B]
        bias = (self.user_bias(users).squeeze()
                + self.item_bias(items).squeeze())
        return dot + bias                        # raw score


mf_model = MatrixFactorization(n_users, n_items, n_factors=32).to(device)
print(f'MF parameters: {sum(p.numel() for p in mf_model.parameters()):,}')

## 6. Neural Collaborative Filtering

MF forces the user-item interaction to be a dot product — a linear combination. NCF replaces that with an MLP, which can learn non-linear interactions:

```
x     = concat(user_emb[u], item_emb[i])   # [2 * n_factors]
r_hat = MLP(x)                              # learned non-linear score
```

The trade-off: more expressive, but also more parameters and slower to train. In practice, the best systems combine both (dot product + MLP in parallel, then merge).

In [ ]:
class NeuralCF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32, hidden=128):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        self.mlp = nn.Sequential(
            nn.Linear(n_factors * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2),   nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, users, items):
        u = self.user_emb(users)              # [B, n_factors]
        v = self.item_emb(items)              # [B, n_factors]
        x = torch.cat([u, v], dim=1)          # [B, n_factors * 2]
        return self.mlp(x).squeeze()          # [B]


ncf_model = NeuralCF(n_users, n_items, n_factors=32, hidden=128).to(device)
print(f'NCF parameters: {sum(p.numel() for p in ncf_model.parameters()):,}')

## 7. Training

We use MSE loss and report **RMSE** (root mean squared error) — same units as the ratings (stars). A naive baseline that always predicts the mean rating (~3.5) gets about RMSE 1.1. A good model should get below 0.95.

In [ ]:
def train_epoch_rec(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for users, items, ratings_batch in loader:
        users, items, ratings_batch = users.to(device), items.to(device), ratings_batch.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss = criterion(preds, ratings_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(ratings_batch)
    return (total_loss / len(loader.dataset)) ** 0.5  # RMSE


@torch.no_grad()
def evaluate_rec(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    for users, items, ratings_batch in loader:
        users, items, ratings_batch = users.to(device), items.to(device), ratings_batch.to(device)
        preds = model(users, items)
        total_loss += criterion(preds, ratings_batch).item() * len(ratings_batch)
    return (total_loss / len(loader.dataset)) ** 0.5  # RMSE

In [ ]:
criterion = nn.MSELoss()

mf_optimizer  = optim.Adam(mf_model.parameters(),  lr=1e-2)
ncf_optimizer = optim.Adam(ncf_model.parameters(), lr=1e-3)

EPOCHS = 20
mf_history  = {'train': [], 'test': []}
ncf_history = {'train': [], 'test': []}

print('Training MF and NCF in parallel...')
for epoch in range(1, EPOCHS + 1):
    mf_train  = train_epoch_rec(mf_model,  train_loader, mf_optimizer,  criterion)
    mf_test   = evaluate_rec(mf_model,  test_loader, criterion)
    ncf_train = train_epoch_rec(ncf_model, train_loader, ncf_optimizer, criterion)
    ncf_test  = evaluate_rec(ncf_model, test_loader, criterion)

    mf_history['train'].append(mf_train);   mf_history['test'].append(mf_test)
    ncf_history['train'].append(ncf_train); ncf_history['test'].append(ncf_test)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:02d} | MF  train: {mf_train:.4f}, test: {mf_test:.4f} | '
              f'NCF train: {ncf_train:.4f}, test: {ncf_test:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, mf_history['train'],  label='MF train')
ax1.plot(epochs, mf_history['test'],   label='MF test')
ax1.plot(epochs, ncf_history['train'], label='NCF train', linestyle='--')
ax1.plot(epochs, ncf_history['test'],  label='NCF test',  linestyle='--')
ax1.axhline(y=1.1, color='gray', linestyle=':', label='naive baseline (~1.1)')
ax1.set_title('RMSE over training')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('RMSE')
ax1.legend()

models = ['MF', 'NCF']
final_test = [mf_history['test'][-1], ncf_history['test'][-1]]
ax2.bar(models, final_test)
ax2.axhline(y=1.1, color='gray', linestyle=':', label='naive baseline')
ax2.set_title('Final test RMSE')
ax2.set_ylabel('RMSE (lower is better)')
ax2.set_ylim(0.8, 1.2)
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Final MF  test RMSE: {mf_history["test"][-1]:.4f}')
print(f'Final NCF test RMSE: {ncf_history["test"][-1]:.4f}')
print('Naive baseline (always predict mean rating): ~1.1 RMSE')

## 8. Top-K Recommendations

At inference time, we want to answer: "given this user, which movies should we recommend?" We score every movie the user hasn't seen, then return the top K.

In [ ]:
@torch.no_grad()
def recommend(model, user_id_1indexed, top_k=10):
    model.eval()
    uid = user_id_1indexed - 1  # zero-index

    # Score all items for this user
    user_tensor = torch.tensor([uid] * n_items, dtype=torch.long).to(device)
    item_tensor = torch.arange(n_items, dtype=torch.long).to(device)
    scores = model(user_tensor, item_tensor).cpu().numpy()

    # Mask out already-rated movies
    seen_items = set(ratings[ratings.user_id == user_id_1indexed].item_id.values - 1)
    scores[list(seen_items)] = -np.inf

    top_indices = np.argsort(scores)[::-1][:top_k]
    print(f'Top {top_k} recommendations for user {user_id_1indexed} '
          f'(rated {len(seen_items)} movies already):')
    for rank, idx in enumerate(top_indices, 1):
        title = movies[movies.item_id == idx + 1]['title'].values[0]
        print(f'  {rank:2d}. {title}  (predicted score: {scores[idx]:.2f})')


print('--- Matrix Factorization ---')
recommend(mf_model, user_id_1indexed=1)
print()
print('--- Neural CF ---')
recommend(ncf_model, user_id_1indexed=1)

In [ ]:
# What did this user actually rate highly? (for sanity check)
user_ratings = ratings[ratings.user_id == 1].merge(movies, on='item_id')
top_rated = user_ratings.sort_values('rating', ascending=False).head(10)
print('User 1 top-rated movies:')
for _, row in top_rated.iterrows():
    print(f'  {int(row.rating)} stars — {row.title}')

## 9. Visualise Item Embeddings

Just like word embeddings cluster semantically similar words together, item embeddings should cluster similar movies. We can check by projecting the 32-D embeddings down to 2D with PCA.

In [ ]:
from sklearn.decomposition import PCA

with torch.no_grad():
    item_vecs = mf_model.item_emb.weight.cpu().numpy()  # [n_items, 32]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(item_vecs)
print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.1%}')

# A few well-known films to label (item IDs are 1-indexed in movies df)
# ml-100k item 1=Toy Story, 50=Star Wars, 56=Pulp Fiction, 100=Fargo, 181=Return of the Jedi
highlights = {1: 'Toy Story', 50: 'Star Wars', 56: 'Pulp Fiction',
              100: 'Fargo', 181: 'Return of the Jedi'}

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(coords[:, 0], coords[:, 1], alpha=0.2, s=6, color='steelblue')

for item_id_1indexed, name in highlights.items():
    idx = item_id_1indexed - 1
    ax.scatter(coords[idx, 0], coords[idx, 1], s=80, zorder=5, color='crimson')
    ax.annotate(name, coords[idx], fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_title('Item embeddings projected to 2D (PCA)\nSimilar movies should cluster together')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

In [ ]:
# Nearest neighbours in embedding space for a query movie
@torch.no_grad()
def similar_movies(model, item_id_1indexed, top_k=8):
    vecs = model.item_emb.weight.cpu()           # [n_items, n_factors]
    query = vecs[item_id_1indexed - 1]           # [n_factors]
    # Cosine similarity
    vecs_norm  = vecs  / vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
    query_norm = query / query.norm().clamp(min=1e-8)
    sims = (vecs_norm @ query_norm).numpy()      # [n_items]
    sims[item_id_1indexed - 1] = -1              # exclude self
    top_idx = np.argsort(sims)[::-1][:top_k]
    query_title = movies[movies.item_id == item_id_1indexed]['title'].values[0]
    print(f'Movies most similar to "{query_title}" in embedding space:')
    for idx in top_idx:
        title = movies[movies.item_id == idx + 1]['title'].values[0]
        print(f'  {sims[idx]:.3f}  {title}')


similar_movies(mf_model, item_id_1indexed=50)   # Star Wars

## 10. Implicit Feedback and Negative Sampling

Most real recommender data is **implicit** — you know what a user clicked or watched, but not what they would have rated. There are no negatives in the data: absence of a watch doesn't mean dislike, it could just mean the user never saw it.

### The training setup

We treat observed interactions as **positive** (label = 1) and sample random unseen items as **negatives** (label = 0). The model is trained to output high scores for positives and low scores for negatives, using binary cross-entropy loss:

```python
loss = BCEWithLogitsLoss(model(u, i_pos), 1) 
     + BCEWithLogitsLoss(model(u, i_neg), 0)
```

### Why negative sampling matters

Without negatives, the model can trivially achieve zero loss by predicting high scores everywhere. Negative sampling creates the learning signal: "predict this item higher than those random unseen items."

The number of negatives per positive (negative ratio) is a hyperparameter — typically 1:1 to 1:5. More negatives = harder training signal but slower epochs.

### Evaluation with implicit feedback

RMSE doesn't apply anymore. Instead:
- **Hit Rate @ K**: for each test user, does any of their held-out items appear in the top-K predicted?
- **NDCG @ K**: like hit rate but rewards higher-ranked hits more

We won't implement this fully here (it needs a different data split), but it's the standard setup for production recommenders.

In [ ]:
# Implicit feedback training loop (conceptual — uses the same ratings data
# but treats any observed rating as a positive and samples random negatives)

class ImplicitMF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, items):
        u = self.user_emb(users)
        v = self.item_emb(items)
        return (u * v).sum(dim=1)  # raw logit (no sigmoid — BCEWithLogitsLoss handles it)


implicit_model = ImplicitMF(n_users, n_items).to(device)
implicit_optim = optim.Adam(implicit_model.parameters(), lr=1e-2)
bce_loss = nn.BCEWithLogitsLoss()

# All observed (user, item) pairs are positives
pos_users = torch.tensor(ratings.user_id.values - 1, dtype=torch.long)
pos_items = torch.tensor(ratings.item_id.values - 1, dtype=torch.long)

implicit_history = []

for epoch in range(1, 11):
    implicit_model.train()
    # Sample negatives: for each positive, pick a random item
    neg_items = torch.randint(0, n_items, (len(pos_users),))

    # Combine into one batch
    all_users  = torch.cat([pos_users, pos_users])
    all_items  = torch.cat([pos_items, neg_items])
    all_labels = torch.cat([torch.ones(len(pos_users)), torch.zeros(len(pos_users))])

    # Mini-batch SGD
    idx = torch.randperm(len(all_users))
    total_loss = 0.0
    batch_size = 1024
    for start in range(0, len(idx), batch_size):
        b = idx[start:start + batch_size]
        u, i, y = all_users[b].to(device), all_items[b].to(device), all_labels[b].to(device)
        implicit_optim.zero_grad()
        loss = bce_loss(implicit_model(u, i), y)
        loss.backward()
        implicit_optim.step()
        total_loss += loss.item()
    implicit_history.append(total_loss)
    if epoch % 2 == 0:
        print(f'Epoch {epoch:02d} | loss: {total_loss:.4f}')

print('\nImplicit model trained — recommends based on interaction patterns, not ratings')

## Summary

| Concept | Key takeaway |
|---|---|
| Rating matrix | Extremely sparse — the job is filling in the blanks |
| Matrix factorization | User + item `nn.Embedding` + dot product; same math as word vectors |
| Bias terms | Capture per-user and per-item offsets so embeddings learn interaction effects |
| Neural CF | Replace dot product with MLP; more expressive but more parameters |
| Implicit feedback | Treat watched/clicked as positives, sample random negatives; BCE loss not MSE |
| Top-K inference | Score all unseen items, mask rated ones, return top K |
| Embedding geometry | Similar items cluster in embedding space — same intuition as Word2Vec |

**What real systems add on top:**
- **Side features** — user demographics, item genres, context (time of day) fed into the MLP
- **Sequence models** — transformer or RNN over a user's watch history for "what to watch next"
- **Two-stage retrieval** — fast approximate nearest-neighbour search to get candidates, then a heavier re-ranking model
- **Exploration vs exploitation** — occasionally recommend outside a user's comfort zone to learn new preferences

Module 5 (Transformers) builds the sequence modelling piece that many modern recommenders use for the "next item" prediction problem.